# ANNA RL Training - Google Colab

Este notebook entrena un modelo PPO para ANNA usando la infraestructura de Google Colab.

## Requisitos
- Runtime: **GPU** (recomendado) o T4
- ~10GB de espacio en disco

## 1. Clonar repositorio

In [ ]:
import os
os.chdir('/content')

!rm -rf Odisea_Game
!git clone https://github.com/icarito/Odisea.git Odisea_Game
%cd Odisea_Game/src

!git checkout codex/anna-bestof-74plus

## 2. Instalar dependencias Python

In [ ]:
!pip install gymnasium==1.2.3 stable-baselines3 torch numpy

## 3. Instalar Godot 3.6 Headless

Colab no tiene display, necesitamos Godot headless.

In [ ]:
import os
import urllib.request
import zipfile

godot_url = "https://github.com/godotengine/godot/releases/download/3.6-stable/Godot_v3.6-stable_linux_headless.64.zip"
godot_zip = "/tmp/godot_headless.zip"
godot_dir = "/opt/godot"

os.makedirs(godot_dir, exist_ok=True)

print("Downloading Godot 3.6 headless...")
urllib.request.urlretrieve(godot_url, godot_zip)

with zipfile.ZipFile(godot_zip, 'r') as zf:
    zf.extractall(godot_dir)

godot_bin = os.path.join(godot_dir, "Godot_v3.6-stable_linux_headless.64")
os.chmod(godot_bin, 0o755)

os.environ['GODOT_BIN'] = godot_bin
os.environ['PATH'] = godot_dir + ':' + os.environ['PATH']

!{godot_bin} --version

## 4. Instalar xvfb (para virtual framebuffer)

In [ ]:
!apt-get update && apt-get install -y xvfb

## 5. Importar assets de Godot

**CRÍTICO**: Ejecutar Godot una vez para importar todos los assets. Sin esto, las escenas fallan al cargar.

In [ ]:
import os
os.chdir('/content/Odisea_Game/src')
godot_bin = os.environ['GODOT_BIN']

print("Importing Godot assets (this may take 2-3 minutes)...")
!xvfb-run -a {godot_bin} --audio-driver Dummy --path . --import --quit-after 100 2>&1 | tail -20

print("\n=== Import complete ===")
!ls -la .import/ 2>/dev/null | head -5 || echo "No .import folder found"

## 6. Configurar variables de entorno

In [ ]:
import os
os.environ['GODOT_BIN'] = '/opt/godot/Godot_v3.6-stable_linux_headless.64'
os.environ['ANNA_RL_MODE'] = '1'
os.chdir('/content/Odisea_Game/src')
print(f"Working dir: {os.getcwd()}")
print(f"GODOT_BIN: {os.environ['GODOT_BIN']}")

## 7. Entrenar modelo ANNA

Ajusta `TIMESTEPS` según el tiempo disponible.

In [ ]:
TIMESTEPS = 500000  # Ajustar según tiempo disponible
CPU_THREADS = 2     # Colab free tier tiene 2 vCPUs
SEED = 42

!python3 agents/train_anna.py \
    --timesteps {TIMESTEPS} \
    --cpu-threads {CPU_THREADS} \
    --seed {SEED} \
    --model-out agents/models/anna_ppo_colab.zip \
    --tensorboard-log agents/runs/tensorboard_colab \
    --no-launch

## 8. Ver resultados

In [ ]:
import json
from pathlib import Path

meta_path = Path('agents/models/anna_ppo_colab.meta.json')
if meta_path.exists():
    with open(meta_path) as f:
        meta = json.load(f)
    print("=== Training Complete ===")
    print(f"Duration: {meta.get('duration_sec', 0):.1f} seconds")
    print(f"Timesteps: {meta.get('timesteps', 0)}")
    print(f"Model saved: {meta.get('model')}")
else:
    print("No metadata found - training may have failed")

!ls -la agents/models/

## 9. Descargar modelo

Ejecuta esta celda para descargar el modelo entrenado a tu máquina local.

In [ ]:
from google.colab import files

model_path = 'agents/models/anna_ppo_colab.zip'
meta_path = 'agents/models/anna_ppo_colab.meta.json'

if Path(model_path).exists():
    files.download(model_path)
    files.download(meta_path)
else:
    print("Model not found!")

## 10. (Opcional) TensorBoard

Si quieres visualizar el entrenamiento en tiempo real.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir agents/runs/tensorboard_colab